# Categorize Responses in the Behavioural Set

In [1]:
import sys, os

PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(),".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0,PARENT_DIR)

sys.path

['/Users/robertagarcia/Desktop/learning/bert_symptom_ner',
 '/opt/homebrew/Cellar/python@3.13/3.13.3_1/Frameworks/Python.framework/Versions/3.13/lib/python313.zip',
 '/opt/homebrew/Cellar/python@3.13/3.13.3_1/Frameworks/Python.framework/Versions/3.13/lib/python3.13',
 '/opt/homebrew/Cellar/python@3.13/3.13.3_1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload',
 '',
 '/Users/robertagarcia/Desktop/learning/bert_symptom_ner/.venv/lib/python3.13/site-packages']

In [2]:
import torch
import json
from typing import List, Literal, Optional, Union, Dict
from enum import Enum
from collections import Counter
import datetime
from dataclasses import dataclass
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Local imports
from config import settings
from gcp_utils import download_from_gcs
from inference.v01.inference_utils import predict_word_level, word_labels_to_spans
from error_analysis.error_categorization import ErrorCategorizer
from error_analysis.error_taxonomy import BehaviouralExample

VERSION = "v02"
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1" 
RUN_IDX = 2
INFERENCE_PIPELINE_VERSION = "v01"

/Users/robertagarcia/Desktop/learning/bert_symptom_ner/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
## Load Required Data Files

# Load id2label mapping
print("📂 Loading id2label mapping...")
with open("../v01/data/id2label.json", "r") as f:
    id2label = json.load(f)

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}

print(f"✅ Loaded {len(id2label)} label mappings")
print(f"Labels: {id2label}")

# Load behavioural evaluation set
print("\n📂 Loading behavioural evaluation set...")
with open("../v01/behavioural_set.json", "r") as f:
    behavioural_set = json.load(f)

print(f"✅ Loaded {len(behavioural_set)} categories")
print(f"Categories: {list(behavioural_set.keys())}")

# Count total examples
total_examples = sum(len(examples) for examples in behavioural_set.values())
print(f"Total test examples: {total_examples}")

📂 Loading id2label mapping...
✅ Loaded 5 label mappings
Labels: {0: 'B-SYMPTOM_NEG', 1: 'B-SYMPTOM_POS', 2: 'I-SYMPTOM_NEG', 3: 'I-SYMPTOM_POS', 4: 'O'}

📂 Loading behavioural evaluation set...
✅ Loaded 7 categories
Categories: ['Core symptom mention (clean baseline)', 'Temporal + progression (very important clinically)', 'Negation & uncertainty (classic failure mode)', 'Multiple symptoms in one sentence (boundary stress test)', 'Long, realistic clinical sentences (THIS IS GOLD 🥇)', 'Attribution & causal language (often tricky)', '“Should NOT extract” edge cases (behavioral guardrails)']
Total test examples: 34


In [5]:
# Load model from the local directory
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()
print("✅ Model is ready to be used")

Using device: mps
✅ Model is ready to be used


## Load the Behavioural Set

In [6]:
# Fit the behavioural set into the dataclasses
for k,examples in behavioural_set.items():
    for i,ex in enumerate(examples):
        behavioural_set[k][i] = BehaviouralExample(**ex)
behavioural_set


{'Core symptom mention (clean baseline)': [BehaviouralExample(example='Patient reports mouth symptom.', entities_with_labels=[{'ent': 'mouth symptom', 'label': 'SYMPTOM_POS', 'start': 16, 'end': 29}]),
  BehaviouralExample(example='The main complaint today is clonic seizure.', entities_with_labels=[{'ent': 'clonic seizure', 'label': 'SYMPTOM_POS', 'start': 28, 'end': 42}]),
  BehaviouralExample(example='Complains of bradypnea since yesterday.', entities_with_labels=[{'ent': 'bradypnea', 'label': 'SYMPTOM_POS', 'start': 13, 'end': 22}]),
  BehaviouralExample(example='Experiencing dysphonia intermittently.', entities_with_labels=[{'ent': 'dysphonia', 'label': 'SYMPTOM_POS', 'start': 13, 'end': 22}]),
  BehaviouralExample(example='Presents with joint inflammation.', entities_with_labels=[{'ent': 'joint inflammation', 'label': 'SYMPTOM_POS', 'start': 14, 'end': 32}])],
 'Temporal + progression (very important clinically)': [BehaviouralExample(example='Patient reports localized superficial 

## Instantiate the Error Categorizer

In [7]:
error_categorizer = ErrorCategorizer()

## Single Sample Example

In [8]:
print("Working example:")
my_example = behavioural_set['Long, realistic clinical sentences (THIS IS GOLD 🥇)'][0]
text = my_example.example
print(f"\t{text}")
tokens, token_labels, word_ids, words, word_labels, word_offsets = predict_word_level(
        text=text,
        model=model,
        tokenizer=tokenizer,
        id2label=id2label,
        device=device,
    )
spans = word_labels_to_spans(text=text, word_offsets=word_offsets, word_labels=word_labels)
print("SPANS:")
for s in spans:
    print(f"\t{s}")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Working example:
	The patient reports lymphadenopathy that started approximately three days ago after physical exertion, associated with mild fatigue but no fever or chills.
SPANS:
	{'start': 0, 'end': 3, 'text': 'The', 'label': 'O'}
	{'start': 4, 'end': 11, 'text': 'patient', 'label': 'O'}
	{'start': 12, 'end': 19, 'text': 'reports', 'label': 'O'}
	{'start': 20, 'end': 101, 'text': 'lymphadenopathy that started approximately three days ago after physical exertion', 'label': 'SYMPTOM_POS'}
	{'start': 101, 'end': 102, 'text': ',', 'label': 'O'}
	{'start': 103, 'end': 113, 'text': 'associated', 'label': 'O'}
	{'start': 114, 'end': 118, 'text': 'with', 'label': 'O'}
	{'start': 119, 'end': 138, 'text': 'mild fatigue but no', 'label': 'SYMPTOM_POS'}
	{'start': 139, 'end': 147, 'text': 'fever or', 'label': 'SYMPTOM_NEG'}
	{'start': 148, 'end': 154, 'text': 'chills', 'label': 'CONFLICT-I-SYMPTOM_NEG-I-SYMPTOM_POS'}
	{'start': 154, 'end': 155, 'text': '.', 'label': 'O'}


*Snipped to check if the expected entities are in the returned spans*

In [9]:
my_example.entities_with_labels

[{'ent': 'lymphadenopathy', 'label': 'SYMPTOM_POS', 'start': 20, 'end': 35},
 {'ent': 'fatigue', 'label': 'SYMPTOM_POS', 'start': 124, 'end': 131},
 {'ent': 'fever', 'label': 'SYMPTOM_NEG', 'start': 139, 'end': 144},
 {'ent': 'chills', 'label': 'SYMPTOM_NEG', 'start': 148, 'end': 154}]

In [17]:
# Example to show how error_categorizer._is_entity_in_spans(a['ent'], spans) works
sample_entity_data = my_example.entities_with_labels[0]
ent = sample_entity_data['ent']
idx = error_categorizer._is_entity_in_spans(ent, spans)
print(f"The entity {ent} was found in the following span: {spans[idx]}")
# See which entities from the behavioural examples are in:
# For present entities, we also track which span index they were found in
present_with_indices = [(e, error_categorizer._is_entity_in_spans(e["ent"], spans)) for e in my_example.entities_with_labels]
present = [(e,idx) for e, idx in present_with_indices if idx is not None]
missing = [(e,idx) for e, idx in present_with_indices if idx is None]

print(f"present_with_indices:\n\t{present_with_indices}")
print(f"present:\n\t{present}")
print(f"missing:\n\t{missing}")

The entity lymphadenopathy was found in the following span: {'start': 20, 'end': 101, 'text': 'lymphadenopathy that started approximately three days ago after physical exertion', 'label': 'SYMPTOM_POS'}
present_with_indices:
	[({'ent': 'lymphadenopathy', 'label': 'SYMPTOM_POS', 'start': 20, 'end': 35}, 3), ({'ent': 'fatigue', 'label': 'SYMPTOM_POS', 'start': 124, 'end': 131}, 7), ({'ent': 'fever', 'label': 'SYMPTOM_NEG', 'start': 139, 'end': 144}, 8), ({'ent': 'chills', 'label': 'SYMPTOM_NEG', 'start': 148, 'end': 154}, 9)]
present:
	[({'ent': 'lymphadenopathy', 'label': 'SYMPTOM_POS', 'start': 20, 'end': 35}, 3), ({'ent': 'fatigue', 'label': 'SYMPTOM_POS', 'start': 124, 'end': 131}, 7), ({'ent': 'fever', 'label': 'SYMPTOM_NEG', 'start': 139, 'end': 144}, 8), ({'ent': 'chills', 'label': 'SYMPTOM_NEG', 'start': 148, 'end': 154}, 9)]
missing:
	[]


*Analyze the present errors for this single example*

In [18]:
errors = error_categorizer._check_entity_detection(
    example=my_example,
    spans=spans
)

present_errors = errors.get("present_errors", [])
missing_entities = errors.get("missing_entities", [])
false_positives = errors.get("false_positives", [])

print("============== ERRORS FOUND ==============")

if present_errors:
    print("\n--- Present errors (detected but wrong boundary/label) ---")
    for p in present_errors:
        e = p['entity']   # ground truth
        s = p['span']     # prediction
        print(f"  Ground Truth : {e['ent']} ({e['label']}) [{e['start']}:{e['end']}]")
        print(f"  Prediction   : {s['text']} ({s['label']}) [{s['start']}:{s['end']}]")
        print(f"  Error(s)     : {p['errors']}\n")

if missing_entities:
    print("\n--- Missing entities (not detected at all) ---")
    for p in missing_entities:
        e = p['entity']   # ground truth only — no predicted span
        print(f"  Ground Truth : {e['ent']} ({e['label']}) [{e['start']}:{e['end']}]")
        print(f"  Prediction   : (not detected)")
        print(f"  Error(s)     : {p['errors']}\n")

if false_positives:
    print("\n--- False positives (predicted but not expected) ---")
    for p in false_positives:
        s = p['span']     # prediction only — no matching ground truth
        print(f"  Prediction   : {s['text']} ({s['label']}) [{s['start']}:{s['end']}]")
        print(f"  Ground Truth : (none — unexpected prediction)")
        print(f"  Error(s)     : {p['errors']}\n")

if not any([present_errors, missing_entities, false_positives]):
    print("  No errors found.")

============== ERRORS FOUND ==============

--- Present errors (detected but wrong boundary/label) ---
  Ground Truth : lymphadenopathy (SYMPTOM_POS) [20:35]
  Prediction   : lymphadenopathy that started approximately three days ago after physical exertion (SYMPTOM_POS) [20:101]
  Error(s)     : ['Boundary Overreach']

  Ground Truth : fatigue (SYMPTOM_POS) [124:131]
  Prediction   : mild fatigue but no (SYMPTOM_POS) [119:138]
  Error(s)     : ['Boundary Overreach']

  Ground Truth : fever (SYMPTOM_NEG) [139:144]
  Prediction   : fever or (SYMPTOM_NEG) [139:147]
  Error(s)     : ['Boundary Overreach']

  Ground Truth : chills (SYMPTOM_NEG) [148:154]
  Prediction   : chills (CONFLICT-I-SYMPTOM_NEG-I-SYMPTOM_POS) [148:154]
  Error(s)     : ['Tokenization Artifacts', 'BIO Sequencing Errors']



# Run Error Categorization thoughout the entire dataset

In [19]:
def _run_through_behavioural_set(
    error_categorizer: ErrorCategorizer,
    behavioural_set: Dict[str, List[BehaviouralExample]],
    model,
    tokenizer,
    id2label: dict,
    device: str,
) -> dict:
    """Run inference + `_check_entity_detection` on every example.

    Clears `error_categorizer.error_counts` so one full sweep has a single aggregate counter.
    Returns dict with `error_counts` (Counter), plus flat lists of error records for inspection.
    """
    error_categorizer.error_counts.clear()

    total_present_errors: List[dict] = []
    total_missing_entities: List[dict] = []
    total_false_positives: List[dict] = []

    for ex_type, examples in behavioural_set.items():
        print(f"Category: {ex_type}")
        for ex in examples:
            text = ex.example
            tokens, token_labels, word_ids, words, word_labels, word_offsets = predict_word_level(
                text=text,
                model=model,
                tokenizer=tokenizer,
                id2label=id2label,
                device=device,
            )
            spans = word_labels_to_spans(
                text=text,
                word_offsets=word_offsets,
                word_labels=word_labels,
            )
            errors = error_categorizer._check_entity_detection(example=ex, spans=spans)
            pe = errors.get("present_errors", [])
            me = errors.get("missing_entities", [])
            fp = errors.get("false_positives", [])
            total_present_errors.extend(pe)
            total_missing_entities.extend(me)
            total_false_positives.extend(fp)
        print(f"  Processed {len(examples)} examples\n")

    return {
        "error_counts": error_categorizer.error_counts,
        "present_errors": total_present_errors,
        "missing_entities": total_missing_entities,
        "false_positives": total_false_positives,
    }


In [20]:
results = _run_through_behavioural_set(
    behavioural_set=behavioural_set,
    error_categorizer=error_categorizer,
    model=model,
    id2label=id2label,
    tokenizer=tokenizer,
    device=device
)

Category: Core symptom mention (clean baseline)
  Processed 5 examples

Category: Temporal + progression (very important clinically)
  Processed 5 examples

Category: Negation & uncertainty (classic failure mode)
  Processed 5 examples

Category: Multiple symptoms in one sentence (boundary stress test)
  Processed 5 examples

Category: Long, realistic clinical sentences (THIS IS GOLD 🥇)
  Processed 5 examples

Category: Attribution & causal language (often tricky)
  Processed 5 examples

Category: “Should NOT extract” edge cases (behavioral guardrails)
  Processed 4 examples



In [21]:
# These results are very important they dictate the next steps for improving the model!
results["error_counts"]

Counter({'Boundary Overreach': 31,
         'Irrelevant Span Mislabeling': 30,
         'Model Overgeneralization': 27,
         'Incorrect Polarity Assignment': 1,
         'Tokenization Artifacts': 1,
         'BIO Sequencing Errors': 1})

In [22]:
results

{'error_counts': Counter({'Boundary Overreach': 31,
          'Irrelevant Span Mislabeling': 30,
          'Model Overgeneralization': 27,
          'Incorrect Polarity Assignment': 1,
          'Tokenization Artifacts': 1,
          'BIO Sequencing Errors': 1}),
 'present_errors': [{'entity': {'ent': 'bradypnea',
    'label': 'SYMPTOM_POS',
    'start': 13,
    'end': 22},
   'span': {'start': 13,
    'end': 38,
    'text': 'bradypnea since yesterday',
    'label': 'SYMPTOM_POS'},
   'span_idx': 2,
   'errors': ['Boundary Overreach']},
  {'entity': {'ent': 'dysphonia',
    'label': 'SYMPTOM_POS',
    'start': 13,
    'end': 22},
   'span': {'start': 13,
    'end': 37,
    'text': 'dysphonia intermittently',
    'label': 'SYMPTOM_POS'},
   'span_idx': 1,
   'errors': ['Boundary Overreach']},
  {'entity': {'ent': 'localized superficial lump',
    'label': 'SYMPTOM_POS',
    'start': 16,
    'end': 42},
   'span': {'start': 16,
    'end': 68,
    'text': 'localized superficial lump that 

How come the results were better when I tried the entire network...?

In [23]:
len("Preserva raciocínio e monitorização, mas troca diazepam por clonazepam e adiciona ondansetrona sem detalhar justificativa.")

122